# Neighborhood bike flows — 8 am vs 8 pm

**Goal**: show how bikes move between neighborhoods during a typical weekday.

- **Choropleth**: each neighborhood coloured by its net bike balance  
  (blue = gains bikes during the day, red = loses bikes).
- **Arrows**: flows from neighborhoods that *lose* bikes to the ones that *gain* them,  
  sized proportionally to the magnitude of the transfer.

**Interpretation**:  
A neighborhood that **loses** bikes 8 am → 8 pm is a *commuting destination*  
(bikes arrive in the morning, leave in the evening).  
A neighborhood that **gains** bikes is a *commuting origin*  
(residents take bikes out in the morning and get them back at night).

In [10]:
import sys
from pathlib import Path

import folium
import branca.colormap as cm
import numpy as np
import pandas as pd

_db_dir = next(
    (p / "src" / "db_upload")
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "db_upload" / "_db.py").exists()
)
sys.path.insert(0, str(_db_dir))
from _db import connect, fetch_all

## 1 · Occupancy at 8 am and 8 pm per station (weekday average)

In [11]:
SQL_OCC = """
SELECT
    station_id,
    AVG(getValue(i)) FILTER (
        WHERE EXTRACT(hour FROM getTimestamp(i) AT TIME ZONE 'Europe/Madrid')::int = 8
    ) AS bikes_8am,
    AVG(getValue(i)) FILTER (
        WHERE EXTRACT(hour FROM getTimestamp(i) AT TIME ZONE 'Europe/Madrid')::int = 20
    ) AS bikes_8pm
FROM station_status_mdb,
LATERAL unnest(instants(bikes_history)) i
WHERE EXTRACT(isodow FROM getTimestamp(i) AT TIME ZONE 'Europe/Madrid') BETWEEN 1 AND 5
  AND EXTRACT(hour   FROM getTimestamp(i) AT TIME ZONE 'Europe/Madrid')::int IN (8, 20)
GROUP BY station_id;
"""

conn = connect()
occ_rows = fetch_all(conn, SQL_OCC)
conn.close()

occ = (
    pd.DataFrame(occ_rows, columns=["station_id", "bikes_8am", "bikes_8pm"])
    .set_index("station_id")
    .astype(float)
    .dropna()
)
print(f"{len(occ)} stations with data at both hours")
occ.head()

549 stations with data at both hours


,bikes_8am,bikes_8pm
station_id,,
184,9.178882,9.986650
87,2.114224,4.722675
477,3.912313,3.079304
273,23.198574,9.567816
394,4.769942,8.036093


## 2 · Station metadata and neighborhood polygons

In [12]:
conn = connect()

sta_rows = fetch_all(conn, """
    SELECT station_id, neighborhood_id, capacity
    FROM stations
    WHERE geom IS NOT NULL AND capacity > 0
""")
stations = (
    pd.DataFrame(sta_rows, columns=["station_id", "neighborhood_id", "capacity"])
    .set_index("station_id")
)

# codi_barri = PK, nom_barri = name
nbh_rows = fetch_all(conn, """
    SELECT
        codi_barri,
        nom_barri,
        ST_AsGeoJSON(geom)          AS geojson,
        ST_Y(ST_Centroid(geom))     AS centroid_lat,
        ST_X(ST_Centroid(geom))     AS centroid_lon,
        income_idx_2022,
        population,
        pop_density_km2
    FROM neighborhoods
    ORDER BY codi_barri;
""")
conn.close()

nbh = pd.DataFrame(
    nbh_rows,
    columns=["codi_barri", "nom_barri", "geojson",
             "centroid_lat", "centroid_lon",
             "income_idx", "population", "pop_density_km2"]
).set_index("codi_barri")

print(f"{len(nbh)} neighborhoods loaded")
nbh[["nom_barri", "centroid_lat", "centroid_lon", "income_idx"]].head()

73 neighborhoods loaded


,nom_barri,centroid_lat,centroid_lon,income_idx
codi_barri,,,,
1,el Raval,41.378999,2.170437,60.94
2,el Barri Gòtic,41.381292,2.177312,83.07
3,la Barceloneta,41.377683,2.190148,78.43
4,"Sant Pere, Santa Caterina i la Ribera",41.386795,2.183431,85.45
5,el Fort Pienc,41.397406,2.181499,105.50


## 3 · Aggregate net balance per neighborhood

In [13]:
df = occ.join(stations, how="inner")

# Occupancy ratio (bikes / capacity)
df["occ_8am"] = df["bikes_8am"] / df["capacity"]
df["occ_8pm"] = df["bikes_8pm"] / df["capacity"]

nbh_agg = df.groupby("neighborhood_id").agg(
    occ_8am    = ("occ_8am",   "mean"),
    occ_8pm    = ("occ_8pm",   "mean"),
    n_stations = ("capacity",  "count"),
).copy()

# Absolute bike delta (sum over all stations in the neighborhood)
nbh_agg["delta_bikes"] = (
    df.groupby("neighborhood_id")["bikes_8pm"].sum()
    - df.groupby("neighborhood_id")["bikes_8am"].sum()
)
nbh_agg["delta_occ"] = nbh_agg["occ_8pm"] - nbh_agg["occ_8am"]

# Attach centroid, name and socioeconomic data
nbh_agg = nbh_agg.join(
    nbh[["nom_barri", "centroid_lat", "centroid_lon", "income_idx", "population", "pop_density_km2"]]
)

print(f"Neighborhoods with Bicing stations: {len(nbh_agg)}")
print("\nTop 5 net senders (lose bikes during the day = commuting destinations):")
print(nbh_agg.nsmallest(5, "delta_bikes")[["nom_barri", "delta_bikes", "n_stations"]].to_string())
print("\nTop 5 net receivers (gain bikes during the day = residential origins):")
print(nbh_agg.nlargest(5, "delta_bikes")[["nom_barri", "delta_bikes", "n_stations"]].to_string())

Neighborhoods with Bicing stations: 64

Top 5 net senders (lose bikes during the day = commuting destinations):
                                         nom_barri  delta_bikes  n_stations
neighborhood_id                                                            
7.0                         la Dreta de l'Eixample   -88.219502          41
66.0             el Parc i la Llacuna del Poblenou   -85.151646          12
21.0                                     Pedralbes   -79.520055           9
3.0                                 la Barceloneta   -49.308766          15
19.0                                     les Corts   -46.085456          17

Top 5 net receivers (gain bikes during the day = residential origins):
                                             nom_barri  delta_bikes  n_stations
neighborhood_id                                                                
10.0                                       Sant Antoni    91.616012          17
4.0              Sant Pere, Santa Caterina i

## 4 · Gravity model flows between neighborhoods

In [14]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


senders   = nbh_agg[nbh_agg["delta_bikes"] < -1].copy()
receivers = nbh_agg[nbh_agg["delta_bikes"] >  1].copy()
senders["outflow"]  = senders["delta_bikes"].abs()
receivers["inflow"] = receivers["delta_bikes"]

flows = []
for s_id, s in senders.iterrows():
    weights = {
        r_id: r.inflow / max(haversine_km(s.centroid_lat, s.centroid_lon,
                                          r.centroid_lat, r.centroid_lon), 0.1)
        for r_id, r in receivers.iterrows()
    }
    total_w = sum(weights.values())
    if total_w == 0:
        continue
    for r_id, w in weights.items():
        flow = s.outflow * (w / total_w)
        if flow >= 0.5:
            flows.append((s_id, r_id, flow))

flows_df = pd.DataFrame(flows, columns=["from_id", "to_id", "flow"])

# Keep top-2 destinations per sender to avoid clutter
TOP_PER_SENDER = 2
flows_top = (
    flows_df
    .sort_values("flow", ascending=False)
    .groupby("from_id")
    .head(TOP_PER_SENDER)
    .reset_index(drop=True)
)
print(f"{len(flows_top)} arrows to draw ({len(senders)} senders, {len(receivers)} receivers)")

53 arrows to draw (36 senders, 20 receivers)


## 5 · Map

In [16]:
# ── helpers ──────────────────────────────────────────────────────────────────

def bezier_curve(p0, p1, n=40, curvature=0.25):
    """Quadratic bezier from p0=[lat,lon] to p1=[lat,lon] with perpendicular offset."""
    p0, p1 = np.array(p0), np.array(p1)
    mid  = (p0 + p1) / 2
    perp = np.array([-(p1[1] - p0[1]), p1[0] - p0[0]])
    ctrl = mid + curvature * perp
    t = np.linspace(0, 1, n)
    curve = np.outer((1-t)**2, p0) + np.outer(2*(1-t)*t, ctrl) + np.outer(t**2, p1)
    return curve.tolist()


def bearing(lat1, lon1, lat2, lon2):
    dlon = np.radians(lon2 - lon1)
    y = np.sin(dlon) * np.cos(np.radians(lat2))
    x = (np.cos(np.radians(lat1)) * np.sin(np.radians(lat2))
         - np.sin(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.cos(dlon))
    return (np.degrees(np.arctan2(y, x)) + 360) % 360


def arrowhead_icon(bearing_deg, color):
    return folium.DivIcon(
        html=(
            f'<div style="transform:rotate({bearing_deg:.0f}deg);width:0;height:0;'
            f'border-left:5px solid transparent;border-right:5px solid transparent;'
            f'border-bottom:13px solid {color};"></div>'
        ),
        icon_size=(10, 13), icon_anchor=(5, 6),
    )


# ── colour scale ─────────────────────────────────────────────────────────────
vmax = nbh_agg["delta_occ"].abs().quantile(0.95)
colorscale = cm.LinearColormap(
    ["#d73027", "#fee090", "#4575b4"],
    vmin=-vmax, vmax=vmax,
    caption="Net occupancy change 8 am → 8 pm  (red = loses bikes, blue = gains bikes)",
)

# ── base map ─────────────────────────────────────────────────────────────────
m = folium.Map(location=[41.39, 2.15], zoom_start=13, tiles="CartoDB positron")
colorscale.add_to(m)

# ── neighborhood choropleth ───────────────────────────────────────────────────
for nbh_id, row in nbh.iterrows():
    if nbh_id not in nbh_agg.index:
        continue
    agg  = nbh_agg.loc[nbh_id]
    color = colorscale(agg["delta_occ"])
    d_str = f"+{agg['delta_occ']*100:.1f}%" if agg["delta_occ"] >= 0 else f"{agg['delta_occ']*100:.1f}%"

    folium.GeoJson(
        data=row["geojson"],
        style_function=lambda _, c=color: {
            "fillColor": c, "color": "white",
            "weight": 0.8, "fillOpacity": 0.65,
        },
        tooltip=f"{row['nom_barri']}: {d_str}  ({int(agg['n_stations'])} stations)",
    ).add_to(m)

    # Centroid label
    folium.Marker(
        location=[agg["centroid_lat"], agg["centroid_lon"]],
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:7px;font-weight:bold;color:#111;'
                f'white-space:nowrap;text-shadow:0 0 3px #fff,0 0 3px #fff">'
                f'{row["nom_barri"]}<br>'
                f'<span style="font-size:8px;color:{"#1a5276" if agg["delta_occ"]>=0 else "#922b21"}">{d_str}</span>'
                f'</div>'
            ),
            icon_size=(130, 28), icon_anchor=(65, 14),
        ),
    ).add_to(m)

# ── flow arrows ──────────────────────────────────────────────────────────────
max_flow = flows_top["flow"].max()
ARROW_COLOR = "#922b21"

for _, edge in flows_top.iterrows():
    s = nbh_agg.loc[edge.from_id]
    r = nbh_agg.loc[edge.to_id]
    p0 = [s.centroid_lat, s.centroid_lon]
    p1 = [r.centroid_lat, r.centroid_lon]

    width = 1.5 + 5.5 * (edge.flow / max_flow)

    folium.PolyLine(
        locations=bezier_curve(p0, p1),
        color=ARROW_COLOR, weight=width, opacity=0.7,
        tooltip=f"{s['nom_barri']} → {r['nom_barri']}: {edge.flow:.1f} bikes",
    ).add_to(m)

    folium.Marker(
        location=p1,
        icon=arrowhead_icon(bearing(*p0, *p1), ARROW_COLOR),
    ).add_to(m)

m.save("bicing_flows.html")

## 6 · Summary table

In [17]:
summary = nbh_agg[[
    "nom_barri", "n_stations",
    "occ_8am", "occ_8pm", "delta_bikes", "delta_occ",
    "income_idx", "population",
]].copy()
summary["occ_8am"]    = summary["occ_8am"].map("{:.1%}".format)
summary["occ_8pm"]    = summary["occ_8pm"].map("{:.1%}".format)
summary["delta_bikes"] = summary["delta_bikes"].round(1)
summary["delta_occ"]   = summary["delta_occ"].map("{:+.1%}".format)
summary = summary.sort_values("delta_bikes")
summary

,nom_barri,n_stations,occ_8am,occ_8pm,delta_bikes,delta_occ,income_idx,population
neighborhood_id,,,,,,,,
7.0,la Dreta de l'Eixample,41,32.1%,23.2%,-88.2,-9.0%,128.93,45667
66.0,el Parc i la Llacuna del Poblenou,12,52.8%,28.4%,-85.2,-24.4%,100.65,17603
21.0,Pedralbes,9,32.1%,6.3%,-79.5,-25.8%,166.37,12649
3.0,la Barceloneta,15,57.6%,47.6%,-49.3,-10.0%,78.43,14749
19.0,les Corts,17,19.0%,10.0%,-46.1,-9.0%,126.28,47057
...,...,...,...,...,...,...,...,...
2.0,el Barri Gòtic,12,50.1%,59.5%,30.1,+9.4%,83.07,27878
9.0,la Nova Esquerra de l'Eixample,18,17.2%,25.4%,38.6,+8.2%,113.57,59122
31.0,la Vila de Gràcia,14,6.0%,19.1%,45.6,+13.1%,107.80,51833
